# Debugging a Real-Time Geo Search with Redis
### From 50ms to 10ms Latency

A real-world investigation into optimizing geo queries using Redis Search.

## The Problem

The customer was performing **geo lookups in their primary database** to find nearby drivers/agents.

Current system performance:

- ~100 queries per second
- **p99 latency ≈ ~50 ms**

However during **peak traffic (2–3× load)**:

- p99 latency increased to **~90-150 ms**
- the system struggled to handle the load
- the team had to **manually scale the database infrastructure**

This approach had two key problems:

- Geo queries were putting pressure on the **primary transactional database**
- The system **did not scale smoothly during traffic spikes**

The customer wanted to explore whether **Redis could be used to handle these geo lookups faster and more efficiently**.

## Driver Document Structure

Each driver is stored as a Redis JSON.

In [1]:
import json

with open('config.json') as f:
    config = json.load(f)
    REDIS_HOST = config['redis']['host']
    REDIS_PORT = config['redis']['port']
    REDIS_USERNAME = config['redis']['username']
    REDIS_PASSWORD = config['redis']['password']

In [170]:
import redis
r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, password=REDIS_PASSWORD, decode_responses=True)

r.json().get('driver:123', '.')

{'driver_id': '100123',
 'vehicle_id': 1123,
 'geo_region_id': '5',
 'is_online': False,
 'driver_rating': 4.8,
 'location': '77.632425,12.954699',
 'vehicle': {'type': 'hatchback', 'boot_space': False},
 'cell_id': 'cell_1437_8397'}

### Field Explanation

- `driver_id` → unique identifier for the driver
- `geo_region_id` → logical region or city zone
- `is_online` → whether the driver is currently available
- `driver_rating` → customer rating used for ranking or filtering
- `location` → driver's current GPS coordinates
- `vehicle` → nested object containing vehicle metadata

The nested vehicle field is useful because Redis Search can also index and filter on nested JSON attributes, for example:

- only sedans
- boot_space

specific vehicle categories

## RediSearch Index

```
FT.CREATE idx:drivers ON JSON PREFIX 1 "driver:" SCHEMA \
$.driver_id AS id TAG SORTABLE UNF \
$.geo_region_id AS geo_region_id TAG SORTABLE UNF \
$.is_online AS is_online TAG SORTABLE UNF \
$.cell_id AS cell_id TAG \
$.vehicle.type AS type TAG SORTABLE UNF \
$.vehicle.boot_space AS boot_space TAG SORTABLE UNF \
$.location AS location GEO SORTABLE
```

## The Driver Discovery Query

In [52]:
query_1 = [
    "ft.aggregate",
    "idx:drivers",
    "@geo_region_id:{3} @is_online:{true} @type:{xl} @boot_space:{true} @location:[77.616226 12.906066 5 km]",
    "load", "2", "@id", "@location", "APPLY",
    "geodistance(@location, 77.616226, 12.906066)", "AS", "dist",
    "SORTBY", "2", "@dist", "ASC",
    "LIMIT", "0", "10", "DIALECT", "2"
]

r.execute_command(*query_1)

[684,
 ['id', '222858', 'location', '77.615082,12.906613', 'dist', '138.15'],
 ['id', '313758', 'location', '77.613465,12.907117', 'dist', '321.35'],
 ['id', '283776', 'location', '77.613498,12.907943', 'dist', '362.02'],
 ['id', '165315', 'location', '77.617249,12.902787', 'dist', '381.2'],
 ['id', '214338', 'location', '77.620512,12.906600', 'dist', '468.45'],
 ['id', '118957', 'location', '77.612372,12.904034', 'dist', '475.05'],
 ['id', '149150', 'location', '77.615597,12.910355', 'dist', '481.9'],
 ['id', '108528', 'location', '77.620577,12.904947', 'dist', '487.86'],
 ['id', '199713', 'location', '77.611324,12.905696', 'dist', '533.05'],
 ['id', '108438', 'location', '77.612249,12.902908', 'dist', '556.14']]

## Baseline Latency

In [43]:
import time

def run_query(r_query):
    start = time.time()
    r.execute_command(*r_query)
    return (time.time() - start) * 1000

In [47]:
latencies = [run_query(query_1) for _ in range(200)]

print('avg:', sum(latencies)/len(latencies))
print('p99:', sorted(latencies)[int(len(latencies)*0.99)])
print('max:', max(latencies))

avg: 43.93092155456543
p99: 51.0101318359375
max: 52.74701118469238


## Query Profiling

In [49]:
r.execute_command(
    "ft.profile",
    "idx:drivers",
    "aggregate", "query",
    "@geo_region_id:{3} @is_online:{true} @type:{xl} @boot_space:{true} @location:[77.616226 12.906066 5 km]",
    "load", "2", "@id", "@location", "APPLY",
    "geodistance(@location, 77.616226, 12.906066)", "AS", "dist",
    "SORTBY", "2", "@dist", "ASC",
    "LIMIT", "0", "10", "DIALECT", "2"
)

[[703,
  ['id', '268820', 'location', '77.616956,12.906912', 'dist', '122.96'],
  ['id', '225347', 'location', '77.613507,12.906754', 'dist', '304.55'],
  ['id', '133441', 'location', '77.619500,12.908812', 'dist', '468.27'],
  ['id', '137469', 'location', '77.611435,12.906287', 'dist', '520'],
  ['id', '213383', 'location', '77.620113,12.903008', 'dist', '541.55'],
  ['id', '274648', 'location', '77.618457,12.901706', 'dist', '541.92'],
  ['id', '191633', 'location', '77.621618,12.907359', 'dist', '602.01'],
  ['id', '287003', 'location', '77.622209,12.906441', 'dist', '649.99'],
  ['id', '140881', 'location', '77.617114,12.899976', 'dist', '684.18'],
  ['id', '311902', 'location', '77.622656,12.907838', 'dist', '724.44']],
 ['Shards',
  [['Shard ID',
    '3',
    'Total profile time',
    '17.0618',
    'Parsing time',
    '0.181688',
    'Pipeline creation time',
    '0.013965',
    'Total GIL time',
    '17.068714',
    'Warning',
    ['None'],
    'Internal cursor reads',
    1,
 

## Hypothesis

The GEO radius search may be scanning too many candidates.

#### Lets visualize the current driver lookup

![alt text](img_1.png)

Ideas:

- Reduce the radius
- Pre-filter drivers using **cell IDs**.

#### Lets see how cells/grids looks like in out map

![alt text](img_2.png)

#### Turning Geo-spacial lookup into a grid/cell based lookup

![alt text](img_3.png)

## Cell Filtering Attempt

In [7]:
import math

In [8]:
def get_cell_ids(lat, lon):
    # 1 km in degrees
    cell_size_lat = 1 / 111.0  # ≈ 0.009009°
    cell_size_lon = 1 / (111.0 * math.cos(math.radians(lat)))

    cell_x = math.floor(lat / cell_size_lat)
    cell_y = math.floor(lon / cell_size_lon)

    return cell_x, cell_y


def build_driver_query_with_cell_ids(lon, lat, radius_km=5, cell_size_km=1):
    # --- Compute center cell ---
    cell_x, cell_y = get_cell_ids(lat, lon)

    # --- Number of neighboring cells to cover radius ---
    cell_range = math.ceil(radius_km / cell_size_km)

    cells = []

    for dx in range(-cell_range, cell_range + 1):
        for dy in range(-cell_range, cell_range + 1):
            cells.append(f"cell_{cell_x + dx}_{cell_y + dy}")

    # Build cell filter string
    cell_filter = "@cell_id:{" + "|".join(cells) + "}"
    filter = '@geo_region_id:{3} @is_online:{true} @type:{xl} @boot_space:{true} '
    filter += f'{cell_filter}'

    query = [
        "ft.aggregate",
        "idx:drivers",
        filter,
        "load", "2", "@id", "@location", "APPLY",
        f"geodistance(@location, {lon}, {lat})", "AS", "dist",
        "SORTBY", "2", "@dist", "ASC",
        "LIMIT", "0", "10", "DIALECT", "2"
    ]

    return query

In [51]:
lon, lat, radius_km = 77.616226, 12.906066, 5
query_2 = build_driver_query_with_cell_ids(lon, lat, radius_km)

r.execute_command(*query_2)

[1790,
 ['id', '214399', 'location', '77.615392,12.906461', 'dist', '100.53'],
 ['id', '268820', 'location', '77.616956,12.906912', 'dist', '122.96'],
 ['id', '245287', 'location', '77.616496,12.908068', 'dist', '224.59'],
 ['id', '299963', 'location', '77.613944,12.904829', 'dist', '283.09'],
 ['id', '225347', 'location', '77.613507,12.906754', 'dist', '304.55'],
 ['id', '158574', 'location', '77.614298,12.903793', 'dist', '328.04'],
 ['id', '214858', 'location', '77.619444,12.905846', 'dist', '349.74'],
 ['id', '310023', 'location', '77.613046,12.904588', 'dist', '381.95'],
 ['id', '122931', 'location', '77.612071,12.906517', 'dist', '453.25'],
 ['id', '133441', 'location', '77.619500,12.908812', 'dist', '468.27']]

Run the benchmark again.

In [48]:
latencies = [run_query(query_2) for _ in range(200)]

print('avg:', sum(latencies)/len(latencies))
print('p99:', sorted(latencies)[int(len(latencies)*0.99)])
print('max:', max(latencies))

avg: 37.31680512428284
p99: 51.18584632873535
max: 52.83689498901367


## Why Did Performance Improve?

To improve performance, we changed how the search space was reduced.

Originally the query relied on a **GEO radius lookup**:
```
@location:[lon lat 5 km]
```

While Redis handles GEO queries efficiently, this operation still requires the engine to **evaluate distance calculations for many candidate drivers** before filtering the final results.

When the dataset becomes large, the GEO filter can become one of the **most expensive parts of the query**.

### Optimization Strategy

Instead of directly running a GEO radius search, we introduced a **grid-based spatial partitioning strategy** using `cell_id`.

The map is divided into **small geographic cells**, and each driver is assigned to a cell based on their coordinates.

Example:
```
cell_1427_8392
cell_1427_8393
cell_1427_8394
```


Now, when searching for nearby drivers:

1. Determine the **cells covering the search radius**
2. Filter drivers using **cell_id**
3. Only evaluate a much smaller candidate set

Example query:
```
@geo_region_id:{3}
@is_online:{true}
@type:{xl}
@cell_id:{cell_1427_8392|cell_1427_8393|cell_1427_8394}
```


### Why This Is Faster

This works well because **TAG filters are extremely efficient** in Redis Search.

- `cell_id` filtering performs **direct posting list lookups**
- The engine avoids scanning a large number of drivers
- The candidate set is **significantly smaller before further processing**

In other words:
GEO radius search → compute distance for many drivers
Cell filtering → narrow down candidates using indexed tags


By reducing the number of candidate drivers early in the query pipeline, the overall query execution becomes **faster and more predictable under load**.

## Lessons Learned

1. Redis is more than caching
1. RediSearch can power real-time geo queries
1. Query debugging tools are critical
1. Index cardinality matters
1. Optimization must be measured

## Redis Beyond Caching

Redis is not just a cache.

It can act as a **real-time query engine** for:

- geo search
- full text search
- vector similarity
- real-time analytics